In [1]:
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
from pathlib import Path
from itertools import groupby
from operator import itemgetter
from CustomFunctions import DetailedBalance, utils

In [2]:
####### load common directories and data
time_interval = 10 #sec/frame
basedir = Path('E:/Aaron/Combined_37C_Confocal_PCA_planar/')
datadir = basedir.joinpath('Data_and_Figs')
savedir = basedir.joinpath('Detailed_Balance/')
if not savedir.exists():
    savedir.mkdir()
FullFrame = pd.read_csv(datadir.joinpath('All_Data_with_CGPS_bins.csv'), index_col=0)
centers = pd.read_csv(datadir.joinpath('PC_bin_centers.csv'), index_col=0)
nbins = len(centers.iloc[:,0])
ttot = time_interval * 180
ntrans = 1
bsiter = 3000


## restrict the treatments and PCs specifically for bootstrapping
bstreats = []#['Random','Galvanotaxis', 'DMSO', 'CK666','Para-Nitro-Blebbistatin']
bspcs = []#[[1,2]]
alldatabs = True

In [15]:
# #### all the cgps origins determinned by visual inspection

# #### SHAPE alignment origins
# allorigins = [[[8,8],[8,8],[8,8],[8,8],[8,7],[8,8],[8,8]],
#                 [[8,8],[8,7],[8,8],[8,8],[8,8],[8,8]],
#                     [[7,8],[6,8],[8,8],[8,8],[7,9]],
#                         [[8,8],[8,8],[8,8],[8,8]],
#                             [[8,8],[8,8],[8,8]],
#                                 [[8,8],[8,8]],
#                                     [[8,8]]]


# #### WIDTH alignment origins
# allorigins = [[[8,8],[8,8],[9,8],[8,8],[9,7],[9,8],[9,8]],
#                 [[8,8],[8,8],[8,8],[8,8],[8,8],[8,8]],
#                     [[8,8],[8,8],[8,8],[8,8],[8,8]],
#                         [[8,8],[8,8],[8,8],[8,8]],
#                             [[8,8],[8,8],[8,8]],
#                                 [[6,8],[8,8]],
#                                     [[8,8]]]

#### PLANAR alignment origins
allorigins = [[[7,8],[8,6],[8,7],[8,8],[8,8],[8,8],[9,8]],
                [[8,6],[8,7],[8,8],[8,7],[8,8],[8,8]],
                    [[8,8],[7,8],[8,8],[8,8],[9,8]],
                        [[7,8],[9,8],[8,8],[8,8]],
                            [[8,8],[8,8],[9,8]],
                                [[8,7],[8,8]],
                                    [[8,8]]]

In [3]:
############# create all CGPSs #############

npcs = centers.shape[1]
for a in range(1,npcs+1):
    for b in range(1,npcs+1):
        if a == b:
            continue
        elif savedir.joinpath(f'PC{b}-PC{a}_interpolated_transitions_separated.csv').exists():
            print('Already made this CGPS')
            continue
        else:
            #set the PCs
            whichpcs = [a,b]
            if __name__ ==  '__main__':
                
                ########### get raw transitions and pairs ###########
                rawtrans = DetailedBalance.get_raw_cgps_trajectories(
                        FullFrame, #pandas dataframe with all of the cgps binned data
                        whichpcs, #which two PCs to use in the cgps [x,y]
                        time_interval, #real time between datapoints
                        savedir, #where to save the aggregated trajectories
                        )
                
                ########### interpolate all transitions so that only individual transitions are made ###########
                transdf_sep = DetailedBalance.get_interpolated_cgps_trajectories(
                        rawtrans, #pandas dataframe with all of the cgps binned data
                        whichpcs, #which two PCs to use in the cgps [x,y]
                        savedir, #where to save the aggregated trajectories
                        )

                ############## get the counts of cells leaving 
                trans_rate_df_sep = DetailedBalance.aggregate_transition_counts(
                        transdf_sep, #transdf_sep from get_interpolated_cgps_trajectories
                        whichpcs, #which two PCs to use in the cgps [x,y]
                        savedir, #where to save the aggregated counts
                        nbins, #how many bins in the x and y cgps axes
                        )
                
                
            

Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 2561.8040325216034 minutes
Total time observed in this CGPS was 6323.616507275451 minutes
Total time observed in this CGPS was 1438.3734253163423 minutes
Total time observed in this CGPS was 3152.405812876531 minutes
Total time observed in this CGPS was 35.741651884947714 minutes
Total time observed in this CGPS was 1371.7429756430913 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 2588.7023176773855 minutes
Total time observed in this CGPS was 6384.537937133702 minutes
Total time observed in this CGPS was 1441.2536577570115 minutes
Total time observed in this CGPS was 3193.123257797886 minutes
Total time observed in this CGPS was 35.248663978390724 minutes
Total time observed in this CGPS was 1369.1586076144174 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating traj

Total time observed in this CGPS was 6496.084408244668 minutes
Total time observed in this CGPS was 1450.4346736754453 minutes
Total time observed in this CGPS was 3246.598966372599 minutes
Total time observed in this CGPS was 35.78881988318933 minutes
Total time observed in this CGPS was 1382.8753345870762 minutes
Finished finding transition rates
Already made this CGPS
Already made this CGPS
Already made this CGPS
Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 2569.9717904018516 minutes
Total time observed in this CGPS was 6367.210940957724 minutes
Total time observed in this CGPS was 1442.6668404443228 minutes
Total time observed in this CGPS was 3182.064865432874 minutes
Total time observed in this CGPS was 35.2490713680128 minutes
Total time observed in this CGPS was 1369.2977457512404 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 2588.692436

In [16]:
########### bootstrap transitions and calculate all the aers and cfs around all the pairwise cgps ###############

npcs = centers.shape[1]
for a in range(1,npcs+1):
    for b in range(1,npcs+1):
        #set the PCs
        whichpcs = [a,b]
        #pass if we want to restrict PCs
        if len(bspcs)>0 and whichpcs not in bspcs:
            continue
        ## use a separate savedir to bootstrap using all data
        if alldatabs:
            bssavedir = savedir.joinpath('alldatabs')
            if not bssavedir.exists():
                bssavedir.mkdir()
        else:
            bssavedir = savedir
            
        if a == b:
            continue
        elif bssavedir.joinpath(f'PC{b}-PC{a}_bootstrapped_{ntrans}_transitions.csv').exists():
            print('Already made this plot')
            continue
        else:
            if __name__ ==  '__main__':
                #### open the transitions
                rawtrans = pd.read_csv(savedir.joinpath(
                    f'PC{whichpcs[0]}-PC{whichpcs[1]}_transitions_separated.csv'), index_col=0)
                
                #merge all treatments if bootstrapping with all data
                if alldatabs:
                    rawtrans.loc[:,'Treatment'] = 'alldata'
                    
                #restrict to bootstrapped treatments if desired
                if len(bstreats)>0:
                    rawtrans = rawtrans[rawtrans.Treatment.isin(bstreats)]
                
                
                ############## BOOTSTRAP MANY TRAJECTORIES ##########
                bstrans, bsint, bsframe_sep_full = DetailedBalance.get_bootstrapped_cgps_trajectories(
                        rawtrans, #raw transition pairs from get_raw_cgps_trajectories
                        whichpcs, #which two PCs to use in the cgps [x,y]
                        time_interval, #real time between datapoints
                        bssavedir, #where to save the aggregated counts
                        nbins, #how many bins in the x and y cgps axes
                        ttot, #set the total bootstrap time
                        ntrans, #how many transitions to sample at each step
                        bsiter, #number of times to bootstrap
                        )


                ############# open average bootstrapped currents ###################
                bsfield_sep = DetailedBalance.get_avg_current_error(
                        bsframe_sep_full, #transition rates in the cgps from get_bootstrapped_cgps_trajectories
                        whichpcs, #which two PCs to use in the cgps [x,y]
                        bssavedir, #where to save the aggregated counts
                        nbins, #how many bins in the x and y cgps axes
                        ntrans, #how many transitions to sample at each step
                        )
    
                
                
                ############# measure aer and cycling frequencies ###########
                #add specific scaling
                xyscaling = [centers[f'PC{whichpcs[0]}'].diff().mean(),centers[f'PC{whichpcs[1]}'].diff().mean()]
                #set the origin to the actual center
                center = allorigins[int(a-1)][int(b-(2+a-1))]

                DetailedBalance.get_aer_cf(
                    bstrans,
                    nbins, #how many bins in the x and y cgps axes
                    xyscaling, #scaling of the bins in real units of whatever the CGPS axis parameters are
                    center, #origin in [x bin,y bin]
                    bssavedir, #where to save calculated aers and cfs
                    whichpcs, #which two PCs to use in the cgps [x,y]
                    ntrans, #how many transitions to sample at each step
                    )
                
                

                ############### measure aer and cycling frequency for the raw transitions
                #get the area scaling in x and y based on the size of the bins in the cgps
                results = []
                for i, cell in rawtrans.groupby('CellID'):
                    #sort data and get continuous transitions in order
                    cell, runs = utils.get_consecutive_transitions(cell)
                    for r in runs:
                        c = cell.iloc[r].reset_index(drop=True)
                        results.append(DetailedBalance.get_area_enclosing_rate((
                            c,
                            nbins,
                            xyscaling,
                            center,
                            )))

                #make a dataframe and save it
                allaers = pd.concat(results, ignore_index = True)
                allaers.to_csv(savedir.joinpath(f'PC{whichpcs[0]}-PC{whichpcs[1]}_raw_transition_aer_cf.csv'))



Boostrapping trajectories with 1 transition samples for alldata


100%|██████████| 3000/3000 [03:07<00:00, 16.00it/s]


Interpolating trajectories for alldata


100%|██████████| 3000/3000 [00:05<00:00, 566.87it/s] 


Calculating bootstrapped CGPS transition rates for alldata


...

Calculating bootstrapped CGPS transition rates for alldata


100%|██████████| 3000/3000 [01:29<00:00, 33.62it/s]


Finished bootstrapping


100%|██████████| 3000/3000 [00:04<00:00, 732.02it/s] 


Already made this plot
Already made this plot
Already made this plot
Already made this plot
Already made this plot
Already made this plot
Already made this plot
